

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [ ]:
# !pip install -q langchain-google-genai

import json
from langchain_google_genai import ChatGoogleGenerativeAI

with open("creds.json", "r") as f:
    creds = json.load(f)

api_key = creds["GE_TOKEN"]

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key,
    temperature=0.3,
    max_output_tokens=5000,
)

prompt = """
Explain quantum computing in simple language.
Include this structure:
1. a short definition
2. key advantages
3. current research directions
Use no more than 200 characters total.
"""

response = llm.invoke(prompt)

def print_response(response):
    content = response.content
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                print(block["text"])
    else:
        print(content)
        
print_response(response)

1. Definition: Computers using quantum physics.
2. Key advantages: Massive speed for complex math.
3. Research directions: Reducing errors and improving stability.


### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [18]:
try:
    from langchain_core.prompts import PromptTemplate
except ImportError:
    from langchain.prompts import PromptTemplate

prompt_tmpl = PromptTemplate(
    input_variables=["topic"],
    template=(
        "Explain {topic} in simple language.\n"
        "Include:\n"
        "1) short definition\n"
        "2) key advantages\n"
        "3) current research directions\n"
        "Use no more than 200 characters total."
    )
)

topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI",
]

for topic in topics:
    prompt_text = prompt_tmpl.format(topic=topic)
    response = llm.invoke(prompt_text)
    print(f"\nTopic: {topic}")
    print(50 * "-")
    print_response(response)
    print(50 * "-")


Topic: Баєсівські методи в машинному навчанні
--------------------------------------------------
1) Оновлення ймовірностей через нові дані.
2) Плюси: оцінка невпевненості, робота з малими даними.
3) Напрями: баєсівські нейромережі, варіаційне виведення.
--------------------------------------------------

Topic: Трансформери в машинному навчанні
--------------------------------------------------
1) Архітектура ШІ на базі механізму «уваги».
2) Швидкість навчання та розуміння контексту.
3) Оптимізація обчислень, мультимодальність.
--------------------------------------------------

Topic: Explainable AI
--------------------------------------------------
1) Definition: AI that explains its reasoning to humans.
2) Pros: Increases trust, ensures safety, and reduces bias.
3) Research: Visual tools, causal logic, and transparent-by-design models.
--------------------------------------------------




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [21]:
%pip install -q ddgs

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_anthropic import ChatAnthropic

with open("creds.json", "r") as f:
    creds = json.load(f)

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=creds["ANTHROPIC_API_KEY"],
    temperature=0.3,
    max_tokens=10000,
)

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for information."""
    return search.invoke(query)

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "You are a research assistant. Use the search tool to find 5 recent scientific papers on the given topic. "
        "Search with simple queries like 'artificial intelligence research papers 2025'. "
        "Output ONLY a numbered list:\n"
        "1. Title: <title>\n   Authors: <authors>\n   Description: <1-2 sentence summary>\n"
        "Repeat for papers 2-5. No extra text."
    ),
)

query = "Find 5 recent scientific publications on artificial intelligence from 2025."

try:
    print("Running agent...")
    for step in agent.stream({"messages": [{"role": "user", "content": query}]}, stream_mode="updates"):
        for key, val in step.items():
            print(f"\n[{key}]")
            if "messages" in val:
                for msg in val["messages"]:
                    print(msg)
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")


Running agent...

[model]
content=[{'id': 'toolu_01G1hwtNptopfhdmhS7R4r6o', 'caller': {'type': 'direct'}, 'input': {'query': 'artificial intelligence research papers 2025'}, 'name': 'web_search', 'type': 'tool_use'}] additional_kwargs={} response_metadata={'id': 'msg_01S6mKC7R4MKYBhAedmw8Dfz', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 658, 'output_tokens': 60, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--019d86ac-3d4b-70e1-b049-01172c07fcfb-0' tool_calls=[{'name': 'web_search', 'args': {'query': 'artificial intelligence research papers 2025'}, 'id': 'toolu_01G1hwtNptopfhdmhS7R4r6o', 'type': 'tool_call'}



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [4]:
import io
import json
import contextlib

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_anthropic import ChatAnthropic

with open("creds.json", "r") as f:
    creds = json.load(f)

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=creds["ANTHROPIC_API_KEY"],
    temperature=0.3,
    max_tokens=4096,
)

# Shared namespace so variables persist across execute_python calls
_exec_namespace: dict = {}
search = DuckDuckGoSearchRun()


@tool
def execute_python(code: str) -> str:
    """Execute Python code and return stdout/stderr.
    Use for statistical calculations, trend extrapolation,
    linear regression, and any numeric analysis.
    numpy, scipy, and sklearn are available."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buf), contextlib.redirect_stderr(stderr_buf):
            exec(code, _exec_namespace)
        output = stdout_buf.getvalue()
        errors = stderr_buf.getvalue()
        result = ""
        if output:
            result += f"stdout:\n{output}"
        if errors:
            result += f"\nstderr:\n{errors}"
        return result.strip() or "(no output)"
    except Exception as exc:
        return f"Error: {type(exc).__name__}: {exc}"


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use it to find current inflation rates, Brazil weather / drought reports,
    orange market outlooks, and other economic indicators."""
    return search.invoke(query)


@tool
def finalize_answer(answer: str) -> str:
    """Call this tool ONLY when you have a complete, well-reasoned answer.
    Provide the full response text in 'answer'.
    Calling this tool ends the session."""
    return answer


tools = [execute_python, web_search, finalize_answer]

SYSTEM_PROMPT = (
    "You are a senior business analytics consultant specialising in commodity exports "
    "and agricultural economics.\n\n"
    "When the user asks a forecasting question you MUST:\n"
    "1. Use **web_search** to gather current data (inflation, weather, market demand).\n"
    "2. Use **execute_python** to build a quantitative forecast "
    "(e.g. linear regression, CAGR, and trend extrapolation, other predictive methods).\n"
    "3. Combine the quantitative model with qualitative web-search findings.\n"
    "to quantitative combined result apply factors like inflation, economic outlook, and weather conditions.\n"
    "4. Call **finalize_answer** with your complete, structured answer.\n"
    "   Do NOT call finalize_answer before you have done at least one web search "
    "and one Python calculation.\n\n"
    "Structure your final answer as:\n"
    "- **Historical data summary**\n"
    "- **Quantitative baseline forecast** (number + method)\n"
    "- **Adjustment factors** (inflation, weather, demand) with sources\n"
    "- **Adjusted forecast** for the requested year\n"
    "- **Confidence note**"
)

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

USER_QUESTION = (
    "We export oranges from Brazil. "
    "In 2022, we exported 200t, in 2023 - 190t, in 2024 - 210t, in 2025 - 220t. "
    "Estimate how many oranges we will be able to export in 2026, taking into account "
    "weather conditions in Brazil and the demand for oranges in the world based on the "
    "economic situation."
)

_exec_namespace.clear()
final_answer = None

for step in agent.stream(
    {"messages": [{"role": "user", "content": USER_QUESTION}]},
    stream_mode="updates",
):
    for node_name, val in step.items():
        if "messages" not in val:
            continue
        for msg in val["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"\n[Tool call] {tc['name']}")
                    if tc["name"] == "execute_python":
                        print(f"  code:\n    {tc['args']['code'][:200]}")
                    elif tc["name"] == "web_search":
                        print(f"  query: {tc['args']['query']}")
                    elif tc["name"] == "finalize_answer":
                        print("  (finalizing answer...)")
                        final_answer = tc["args"]["answer"]
            elif hasattr(msg, "name") and msg.name:
                preview = str(msg.content)[:300]
                print(f"\n[{msg.name} result] {preview}")
            elif hasattr(msg, "content") and msg.content and not hasattr(msg, "name"):
                text = msg.content if isinstance(msg.content, str) else str(msg.content)
                if text.strip():
                    print(f"\n[Claude] {text.strip()[:500]}")

print("\n" + "#" * 60)
print(" FINAL ANSWER")
print("#" * 60)
print(final_answer or "(Agent ended without calling finalize_answer)")



[Tool call] web_search
  query: Brazil orange production 2025 2026 weather drought conditions

[Tool call] web_search
  query: global orange market demand 2026 economic outlook

[Tool call] web_search
  query: Brazil citrus production forecast 2026 climate conditions

[Tool call] web_search
  query: world orange juice prices 2025 2026 market trends

[web_search result] Citruscanker affects all varieties ofcitrustrees, and recent outbreaks in Australia,Brazil, and the United States have slowedcitrusproductionin parts of those countries.Citrusleafminer moths are a major concern wherecitruscanker exists. Download Sample Get Special DiscountBrazilCitricAcid Powder Mar

[web_search result] Frozen concentrateorangejuicesegment is projected to hold 25.1% share in the globalorangejuicemarketin2026, largely propelled by its convenience and economic benefits. Discover the latestOrangeJuicepriceforecast with expert technical analysis,trendpredictions, and chart insights. Stay ahead of energ

[we